# Part 5: Business recommendations
**Lab:** In God we trust, everyone else bring the data · **Analyst:** Ayhan Demir · **Notebook 5 of 5**

**Question for the CMO:** how should the $500K be split, and how sure are we?
**Reads:** `marketing_data.csv`, the 3 corrected result tables, the 2 power tables · **Writes:** `executive_memo.md`



### Setup 
- Loads the libraries used in this lab: pandas/numpy for tables, matplotlib/seaborn for charts, `scipy.stats` for the tests.
- `rng` is the fixed random "dice" (seed 42), so simulations and bootstraps give the same numbers every run.
- `OUT` is the `outputs/` folder where every chart, CSV and the memo are saved. `ALPHA = 0.05` is the significance line.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
from pathlib import Path
from IPython.display import Markdown, display

from scipy import stats
from scipy.stats import fisher_exact, false_discovery_control   # false_discovery_control needs SciPy 1.11+

rng = np.random.default_rng(42)          # our "dice": fixed seed so results are repeatable
OUT = Path("outputs"); OUT.mkdir(exist_ok=True)   # all charts / CSV / memo are saved here
sns.set_theme(style="whitegrid")
ALPHA = 0.05

### Load everything from notebooks 01 to 04
- `df` / `paid`: daily data with daily CPA (as in notebook 01).
- `agg`: channel totals with CPA and ROAS (same formulas as Step 3), used for the composite score.
- The BH-corrected result tables (notebook 03) and the power tables (notebook 04), used in the summary and the memo.

In [2]:
df = pd.read_csv(OUT / "marketing_data.csv", parse_dates=["date"])
df["CPA"] = (df.cost / df.conversions).replace([np.inf, -np.inf], np.nan)
paid = df[df.cost > 0]
N_DAYS = df.date.nunique()

num_cols = ["impressions", "clicks", "conversions", "cost", "revenue"]
agg = df.groupby("channel")[num_cols].sum()
agg["conv_rate"] = agg.conversions / agg.clicks
agg["CPA"] = (agg.cost / agg.conversions).where(agg.cost > 0)
agg["ROAS"] = agg.revenue / agg.cost

cpa_res = pd.read_csv(OUT / "results_cpa_ttest.csv")
cvr_t_res = pd.read_csv(OUT / "results_conv_rate_ttest.csv")
fisher_res = pd.read_csv(OUT / "results_conv_rate_fisher.csv")
min_days = pd.read_csv(OUT / "power_min_days.csv")
agg[["cost", "conversions", "conv_rate", "CPA", "ROAS"]].round(3)

,cost,conversions,conv_rate,CPA,ROAS
channel,,,,,
abc,131.660,618,0.418,0.213,268.356
def,15579.244,26716,0.154,0.583,92.100
ghi,151063.181,331992,0.267,0.455,107.458
jkl,8115.213,19200,0.260,0.423,123.940
mno,96912.287,225991,0.235,0.429,121.958


## Step 8: From statistics to a decision

**8a. Summary of significant findings (BH-corrected)**
- Only pairs that are still significant **after BH correction** (notebook 03) are listed, as the template says.
- **CPA:** which channel is cheaper, by how many $ per conversion, the adjusted p-value and Cohen's d.
- **Conversion rate:** Fisher's test, the right test for right/wrong clicks (§4b). Which channel converts more, by how many percentage points, and the adjusted p-value.

In [3]:
cpa_sig = cpa_res[cpa_res.significant_fdr]
print(f"=== CPA: {len(cpa_sig)} of {len(cpa_res)} pairs differ after BH correction ===")
for r in cpa_sig.itertuples():
    cheap, dear = (r.group_a, r.group_b) if r.mean_a < r.mean_b else (r.group_b, r.group_a)
    print(f"{cheap} is cheaper than {dear} by ${abs(r.diff):.3f} per conversion "
          f"| adj. p = {r.p_value_fdr:.2e} | d = {abs(r.cohens_d):.2f} ({r.effect})")

fis_sig = fisher_res[fisher_res.significant_fdr]
print(f"\n=== Conversion rate (Fisher): {len(fis_sig)} of {len(fisher_res)} pairs differ after BH correction ===")
for r in fis_sig.itertuples():
    high, low = (r.group_a, r.group_b) if r.rate_a > r.rate_b else (r.group_b, r.group_a)
    print(f"{high} converts higher than {low} by {abs(r.diff_pp):.1f} points | adj. p = {r.p_value_fdr:.2e}")

=== CPA: 7 of 10 pairs differ after BH correction ===
abc is cheaper than def by $0.253 per conversion | adj. p = 2.09e-06 | d = 1.15 (large)
abc is cheaper than ghi by $0.159 per conversion | adj. p = 7.90e-04 | d = 0.83 (large)
abc is cheaper than jkl by $0.148 per conversion | adj. p = 2.01e-03 | d = 0.71 (medium)
abc is cheaper than mno by $0.124 per conversion | adj. p = 3.65e-03 | d = 0.77 (medium)
ghi is cheaper than def by $0.094 per conversion | adj. p = 1.80e-02 | d = 0.37 (small)
jkl is cheaper than def by $0.105 per conversion | adj. p = 1.32e-02 | d = 0.40 (small)
mno is cheaper than def by $0.129 per conversion | adj. p = 7.90e-04 | d = 0.57 (medium)

=== Conversion rate (Fisher): 10 of 10 pairs differ after BH correction ===
abc converts higher than def by 26.5 points | adj. p = 3.59e-129
abc converts higher than ghi by 15.1 points | adj. p = 4.90e-36
abc converts higher than jkl by 15.8 points | adj. p = 1.02e-38
abc converts higher than mno by 18.3 points | adj. p = 4.

**8b. 95% confidence intervals for daily CPA (bootstrap)**
- **Bootstrap** = resample the channel's own days with replacement 1,000 times and take the mean each time. The middle 95% of those means is the range where the true average daily CPA very probably lies (§2a).
- The resampling uses the fast pattern from §9 (`rng.integers` builds all 1,000 resamples at once).
- **Read it like this:** overlapping ranges = we can't separate the channels. A wide range = few days or a noisy channel.

In [4]:
def bootstrap_ci(data, n_bootstrap=1000, ci_level=0.95):
    data = np.asarray(data)
    idx = rng.integers(0, len(data), (n_bootstrap, len(data)))     # resample days with replacement
    means = data[idx].mean(axis=1)                                  # the mean of each resample
    alpha = 1 - ci_level
    low, high = np.percentile(means, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return low, high

# Table: channel, mean daily CPA, 95% CI low, 95% CI high
rows = []
for ch, grp in paid.groupby("channel"):
    cpa = grp.CPA.dropna()
    low, high = bootstrap_ci(cpa)
    rows.append({"channel": ch, "days": len(cpa), "mean_daily_CPA": cpa.mean(),
                 "ci_low_95": low, "ci_high_95": high})
ci_table = pd.DataFrame(rows).set_index("channel")
ci_table.round(3)

,days,mean_daily_CPA,ci_low_95,ci_high_95
channel,,,,
abc,22,0.257,0.204,0.320
def,91,0.510,0.454,0.569
ghi,91,0.416,0.372,0.461
jkl,91,0.405,0.356,0.461
mno,91,0.381,0.346,0.418


**8c. Composite score and the $500K split**
- **Ranks:** lowest CPA → highest rank, highest ROAS → highest rank. Composite = 50% CPA rank + 50% ROAS rank, using the channel totals from Step 3.
- **Split:** the budget is shared in proportion to the composite score. Each channel is then held between **5% and 35%**: clip to the limits, rescale to 100%, and repeat until it settles.
- `monthly_cost_now` (average monthly spend in the data) is added next to the new budget so the scale jump is visible. You need it for the "can it absorb the budget?" question below.

In [5]:
score = agg.loc[agg.cost > 0, ["cost", "CPA", "ROAS"]].copy()        # paid channels only
score["CPA_rank"] = score.CPA.rank(ascending=False)                  # lowest CPA -> highest rank
score["ROAS_rank"] = score.ROAS.rank()                               # highest ROAS -> highest rank
score["composite"] = 0.5 * score.CPA_rank + 0.5 * score.ROAS_rank
score = score.sort_values("composite", ascending=False)

# Split $500,000 in proportion to the composite score, min 5% and max 35% per channel
BUDGET = 500_000
share = score.composite / score.composite.sum()
for _ in range(20):                          # clip to 5%-35%, rescale to 100%, repeat
    share = share.clip(0.05, 0.35)
    share = share / share.sum()
score["allocation_pct"] = share * 100
score["allocation_usd"] = share * BUDGET
score["monthly_cost_now"] = score.cost / N_DAYS * 30     # average monthly spend in the data
print(f"Total allocated: ${score.allocation_usd.sum():,.0f}")
score[["CPA", "ROAS", "CPA_rank", "ROAS_rank", "composite", "allocation_pct", "allocation_usd", "monthly_cost_now"]].round(2)

Total allocated: $500,000


,CPA,ROAS,CPA_rank,ROAS_rank,composite,allocation_pct,allocation_usd,monthly_cost_now
channel,,,,,,,,
abc,0.21,268.36,5.0,5.0,5.0,33.33,166666.67,43.40
jkl,0.42,123.94,4.0,4.0,4.0,26.67,133333.33,2675.35
mno,0.43,121.96,3.0,3.0,3.0,20.00,100000.00,31949.11
ghi,0.46,107.46,2.0,2.0,2.0,13.33,66666.67,49801.05
def,0.58,92.10,1.0,1.0,1.0,6.67,33333.33,5136.01


**✍️ My answers:**
- **Not significant, yet different ranks: is that fair?** No. `ghi`, `jkl` and `mno` can't be told apart on CPA, but ranks 2 to 4 hand them 13%, 20% and 27% of the budget. That gap comes from the rank formula, not from evidence. Within these three, money should follow proven scale, not rank.
- **Can the top channel absorb a big budget?** Very unlikely at the same CPA. `abc` gets the largest share, 33% (**$166,667 a month**), but it spent about **$43 a month** in the data, a jump of about 3,800×. `jkl` would also jump from about $2.7K to $133K a month (about 50×). CPA usually rises as spend scales (limited inventory, audience saturation), so this needs a staged test, not a jump.
- **Significant but too small to matter:** the conversion-rate gaps between ghi, jkl and mno (0.7, 2.5 and 3.2 points). They are "significant" only because Fisher's N is hundreds of thousands of clicks. None of them is a reason to move budget on its own.

**8d. Executive memo**
- Built as Markdown in an f-string, so every number comes **from the results above** rather than being typed in by hand. It is saved to `outputs/executive_memo.md` and shown below.
- It follows the template's sections. The caveats cover all six points the PDF asks for.
- **Correction used:** BH, as Step 8 asks. Where Bonferroni would change a finding, the memo says so.

In [6]:
# Small text blocks built from the results (so every number in the memo comes from the analysis)
n_tests = len(cpa_res) + len(cvr_t_res) + len(fisher_res)
period = f"{df.date.min():%d %b %Y} to {df.date.max():%d %b %Y} ({N_DAYS} days)"
core = ["ghi", "jkl", "mno"]                                          # the three large placements
spend_share = agg.cost / agg.cost.sum() * 100                         # % of today's spend
cv = paid.groupby("channel").CPA.std() / paid.groupby("channel").CPA.mean()   # real day-to-day spread
gap_ghi_jkl = abs(fisher_res.query("group_a == 'ghi' and group_b == 'jkl'").diff_pp.iloc[0])
monthly_now = agg.cost.sum() / N_DAYS * 30                            # today's total monthly spend
core_split = " / ".join(f"{ch} {score.loc[ch, 'allocation_pct']:.0f}%" for ch in core)
core_mix = " / ".join(f"{ch} {agg.loc[ch, 'cost'] / agg.loc[core, 'cost'].sum():.0%}" for ch in core)   # today's spend mix

cpa_lines = []
for r in cpa_sig.itertuples():
    cheap, dear = (r.group_a, r.group_b) if r.mean_a < r.mean_b else (r.group_b, r.group_a)
    note = "" if r.significant_bonferroni else "; not significant under Bonferroni"
    cpa_lines.append(f"    - {cheap} cheaper than {dear} by ${abs(r.diff):.2f} per conversion "
                     f"(adj. p = {r.p_value_fdr:.1e}, d = {abs(r.cohens_d):.2f}, {r.effect}{note})")
cpa_lines = "\n".join(cpa_lines)

alloc_rows = "\n".join(
    f"| {ch} | {r.CPA:.2f} | {r.ROAS:.0f}x | {r.composite:.1f} | {r.allocation_pct:.1f}% | ${r.allocation_usd:,.0f} | ${r.monthly_cost_now:,.0f} |"
    for ch, r in score.iterrows())
ci_rows = "\n".join(
    f"| {ch} | {r.days:.0f} | {r.mean_daily_CPA:.2f} | {r.ci_low_95:.2f} to {r.ci_high_95:.2f} |"
    for ch, r in ci_table.iterrows())
power_lines = "; ".join(
    f"{r.effect_size_pct}% gap: {r.min_days_for_80pct} days ({r.status_with_our_data})" for r in min_days.itertuples())

# TODO: write the executive memo as Markdown and save it
memo = f"""# Executive memo — Marketing budget allocation

**To:** CMO · **From:** Ayhan Demir, Marketing Analytics · **Date:** {pd.Timestamp.today():%d %b %Y} · **Dataset:** Online Advertising Performance Data (Kaggle; Company X, 5 ad placements: abc, def, ghi, jkl, mno) · **Period:** {period}

## Executive summary
After correcting for {n_tests} statistical tests, the five placements fall into three groups:
- **def is the clear laggard.** Its cost per conversion is {agg.loc['def', 'CPA'] / agg.loc[core, 'CPA'].max() - 1:.0%} to {agg.loc['def', 'CPA'] / agg.loc[core, 'CPA'].min() - 1:.0%} higher than ghi, jkl and mno, and it has the lowest conversion rate ({agg.loc['def', 'conv_rate']:.1%}).
- **ghi, jkl and mno cannot be told apart on CPA** (gaps of 3 to 9%, p > 0.2). Together they carry {spend_share[core].sum():.0f}% of today's spend.
- **abc looks best on every metric** (CPA ${agg.loc['abc', 'CPA']:.2f}, conversion rate {agg.loc['abc', 'conv_rate']:.1%}), **but it is tiny:** ${agg.loc['abc', 'cost']:,.0f} of spend in {N_DAYS} days, with conversions on only {ci_table.loc['abc', 'days']} days. It is promising, but not proven at scale.

**Recommendation:** move budget away from def, keep ghi, jkl and mno as the core, and give abc a capped test budget before scaling it. The formula-based split below over-weights abc and ranks ghi, jkl and mno on differences the data cannot confirm.

## Key findings
- **Top performers:** abc (lowest CPA ${agg.loc['abc', 'CPA']:.2f}, highest ROAS {agg.loc['abc', 'ROAS']:.0f}x, small volume). Among the large channels, jkl and mno have the best CPA (${agg.loc['jkl', 'CPA']:.2f} and ${agg.loc['mno', 'CPA']:.2f}), and ghi has the most conversions ({agg.loc['ghi', 'conversions']:,.0f}) and the highest conversion rate ({agg.loc['ghi', 'conv_rate']:.1%}).
- **Statistically significant differences (after BH correction):**
  - **CPA (Welch t-test on daily CPA): {len(cpa_sig)} of {len(cpa_res)} pairs.**
{cpa_lines}
  - **Conversion rate (Fisher's exact test): {len(fis_sig)} of {len(fisher_res)} pairs.** The ranking is abc > ghi > jkl > mno > def. Several gaps are tiny (ghi vs jkl: {gap_ghi_jkl:.1f} points). They count as significant only because N is in the hundreds of thousands of clicks.
- **Data adequacy (power):** with {N_DAYS} days per channel: {power_lines}. That assumes a day-to-day CPA spread of 15%. The real spread is about {cv.min():.0%} to {cv.max():.0%}, so in practice the gaps between ghi, jkl and mno (3 to 9%) cannot be resolved with this data.

## Recommendations
**Formula-based split** (composite = 50% CPA rank + 50% ROAS rank, min 5% / max 35% per channel):

| Channel | CPA ($) | ROAS | Composite | Share | Monthly budget | Monthly spend today |
|---|---|---|---|---|---|---|
{alloc_rows}

**How to apply it (strategic actions):**
1. **def: keep it at the bottom** ({score.loc['def', 'allocation_pct']:.1f}% in the formula, or down to the 5% floor). It is significantly more expensive than mno under both corrections, more expensive than ghi and jkl under BH, and it has the lowest conversion rate.
2. **ghi, jkl, mno: fund as one core tier.** Their CPA gaps are not significant, so the {core_split} split comes from ranks rather than evidence. Split the core budget by proven scale (today's spend mix: {core_mix}), not by rank. Note that jkl would jump from about ${score.loc['jkl', 'monthly_cost_now']:,.0f} to ${score.loc['jkl', 'allocation_usd']:,.0f} a month.
3. **abc: run a capped scale-up test.** Its formula share (${score.loc['abc', 'allocation_usd']:,.0f} a month) is far above anything it has ever spent (about ${score.loc['abc', 'monthly_cost_now']:,.0f} a month). Raise spend in steps and watch CPA weekly. Move more money in only if CPA holds.
4. **Until abc proves it can scale,** place the unused part of its share with the core tier (ghi, jkl, mno), which already runs at scale.

## Statistical caveats
- **Dataset source and limits:** one advertiser, one ad firm, 3 months in 2020, anonymised placements. "Cost" is what Company X paid the ad firm for the clicks. "Revenue" is post-click sales credited to the ads. A ROAS of {agg.ROAS.min():.0f}x to {agg.ROAS.max():.0f}x points to generous attribution: use ROAS to rank channels, not as a profit forecast. Cleaning removed duplicates, rows with no placement and impossible rows (see notebook 01).
- **Multiple comparisons correction:** {n_tests} tests means about {n_tests * ALPHA:.1f} false positives by chance alone. I used Benjamini-Hochberg. The stricter Bonferroni correction would drop "def is more expensive than ghi and jkl" on CPA, so treat those two as likely rather than proven.
- **Confidence intervals (95% bootstrap, mean daily CPA):**

| Channel | Days | Mean daily CPA ($) | 95% CI ($) |
|---|---|---|---|
{ci_rows}

  The ranges for ghi, jkl and mno overlap. abc's range rests on only {ci_table.loc['abc', 'days']} days.
- **External factors:** April to June 2020 (the early COVID-19 months), the campaign and banner mix differs by placement, and attribution (sales after a click are not proof that the ad caused them). Scale is a factor too: $500K a month is about {BUDGET / monthly_now:.1f}x today's total spend (${monthly_now:,.0f} a month), and CPAs seen at today's spend may not hold at higher spend.
- **Statistical vs practical significance:** all {len(fisher_res)} conversion-rate pairs are significant with Fisher, but the gaps of {gap_ghi_jkl:.1f} to 3.2 points between ghi, jkl and mno are too small to move budget on their own.
- **Power limitations:** the power simulation assumes a 15% day-to-day spread in CPA. The real spread is {cv.min():.0%} to {cv.max():.0%} (3 to 4 times larger), so the true power is lower and "not significant" here means **not enough data**, not "no difference".

## Next steps
1. Agree on the smallest CPA gap worth acting on (for example 10%) **before** the next test.
2. Launch the abc scale-up test with a fixed budget cap and a fixed duration. Review it weekly.
3. To separate ghi, jkl and mno, run a planned test: shift a fixed share of budget between them for a set period and compare against a control.
4. Re-run this analysis monthly with fresh data and re-allocate based on BH-corrected results and confidence intervals.
"""
(OUT / "executive_memo.md").write_text(memo, encoding="utf-8")
display(Markdown(memo))

# Executive memo — Marketing budget allocation

**To:** CMO · **From:** Ayhan Demir, Marketing Analytics · **Date:** 23 Nov 2026 · **Dataset:** Online Advertising Performance Data (Kaggle; Company X, 5 ad placements: abc, def, ghi, jkl, mno) · **Period:** 01 Apr 2020 to 30 Jun 2020 (91 days)

## Executive summary
After correcting for 30 statistical tests, the five placements fall into three groups:
- **def is the clear laggard.** Its cost per conversion is 28% to 38% higher than ghi, jkl and mno, and it has the lowest conversion rate (15.4%).
- **ghi, jkl and mno cannot be told apart on CPA** (gaps of 3 to 9%, p > 0.2). Together they carry 94% of today's spend.
- **abc looks best on every metric** (CPA $0.21, conversion rate 41.8%), **but it is tiny:** $132 of spend in 91 days, with conversions on only 22 days. It is promising, but not proven at scale.

**Recommendation:** move budget away from def, keep ghi, jkl and mno as the core, and give abc a capped test budget before scaling it. The formula-based split below over-weights abc and ranks ghi, jkl and mno on differences the data cannot confirm.

## Key findings
- **Top performers:** abc (lowest CPA $0.21, highest ROAS 268x, small volume). Among the large channels, jkl and mno have the best CPA ($0.42 and $0.43), and ghi has the most conversions (331,992) and the highest conversion rate (26.7%).
- **Statistically significant differences (after BH correction):**
  - **CPA (Welch t-test on daily CPA): 7 of 10 pairs.**
    - abc cheaper than def by $0.25 per conversion (adj. p = 2.1e-06, d = 1.15, large)
    - abc cheaper than ghi by $0.16 per conversion (adj. p = 7.9e-04, d = 0.83, large)
    - abc cheaper than jkl by $0.15 per conversion (adj. p = 2.0e-03, d = 0.71, medium)
    - abc cheaper than mno by $0.12 per conversion (adj. p = 3.7e-03, d = 0.77, medium)
    - ghi cheaper than def by $0.09 per conversion (adj. p = 1.8e-02, d = 0.37, small; not significant under Bonferroni)
    - jkl cheaper than def by $0.11 per conversion (adj. p = 1.3e-02, d = 0.40, small; not significant under Bonferroni)
    - mno cheaper than def by $0.13 per conversion (adj. p = 7.9e-04, d = 0.57, medium)
  - **Conversion rate (Fisher's exact test): 10 of 10 pairs.** The ranking is abc > ghi > jkl > mno > def. Several gaps are tiny (ghi vs jkl: 0.7 points). They count as significant only because N is in the hundreds of thousands of clicks.
- **Data adequacy (power):** with 91 days per channel: 5% gap: 145 days (insufficient); 10% gap: 40 days (sufficient); 15% gap: 20 days (sufficient); 20% gap: 15 days (sufficient). That assumes a day-to-day CPA spread of 15%. The real spread is about 45% to 63%, so in practice the gaps between ghi, jkl and mno (3 to 9%) cannot be resolved with this data.

## Recommendations
**Formula-based split** (composite = 50% CPA rank + 50% ROAS rank, min 5% / max 35% per channel):

| Channel | CPA ($) | ROAS | Composite | Share | Monthly budget | Monthly spend today |
|---|---|---|---|---|---|---|
| abc | 0.21 | 268x | 5.0 | 33.3% | $166,667 | $43 |
| jkl | 0.42 | 124x | 4.0 | 26.7% | $133,333 | $2,675 |
| mno | 0.43 | 122x | 3.0 | 20.0% | $100,000 | $31,949 |
| ghi | 0.46 | 107x | 2.0 | 13.3% | $66,667 | $49,801 |
| def | 0.58 | 92x | 1.0 | 6.7% | $33,333 | $5,136 |

**How to apply it (strategic actions):**
1. **def: keep it at the bottom** (6.7% in the formula, or down to the 5% floor). It is significantly more expensive than mno under both corrections, more expensive than ghi and jkl under BH, and it has the lowest conversion rate.
2. **ghi, jkl, mno: fund as one core tier.** Their CPA gaps are not significant, so the ghi 13% / jkl 27% / mno 20% split comes from ranks rather than evidence. Split the core budget by proven scale (today's spend mix: ghi 59% / jkl 3% / mno 38%), not by rank. Note that jkl would jump from about $2,675 to $133,333 a month.
3. **abc: run a capped scale-up test.** Its formula share ($166,667 a month) is far above anything it has ever spent (about $43 a month). Raise spend in steps and watch CPA weekly. Move more money in only if CPA holds.
4. **Until abc proves it can scale,** place the unused part of its share with the core tier (ghi, jkl, mno), which already runs at scale.

## Statistical caveats
- **Dataset source and limits:** one advertiser, one ad firm, 3 months in 2020, anonymised placements. "Cost" is what Company X paid the ad firm for the clicks. "Revenue" is post-click sales credited to the ads. A ROAS of 92x to 268x points to generous attribution: use ROAS to rank channels, not as a profit forecast. Cleaning removed duplicates, rows with no placement and impossible rows (see notebook 01).
- **Multiple comparisons correction:** 30 tests means about 1.5 false positives by chance alone. I used Benjamini-Hochberg. The stricter Bonferroni correction would drop "def is more expensive than ghi and jkl" on CPA, so treat those two as likely rather than proven.
- **Confidence intervals (95% bootstrap, mean daily CPA):**

| Channel | Days | Mean daily CPA ($) | 95% CI ($) |
|---|---|---|---|
| abc | 22 | 0.26 | 0.20 to 0.32 |
| def | 91 | 0.51 | 0.45 to 0.57 |
| ghi | 91 | 0.42 | 0.37 to 0.46 |
| jkl | 91 | 0.41 | 0.36 to 0.46 |
| mno | 91 | 0.38 | 0.35 to 0.42 |

  The ranges for ghi, jkl and mno overlap. abc's range rests on only 22 days.
- **External factors:** April to June 2020 (the early COVID-19 months), the campaign and banner mix differs by placement, and attribution (sales after a click are not proof that the ad caused them). Scale is a factor too: $500K a month is about 5.6x today's total spend ($89,605 a month), and CPAs seen at today's spend may not hold at higher spend.
- **Statistical vs practical significance:** all 10 conversion-rate pairs are significant with Fisher, but the gaps of 0.7 to 3.2 points between ghi, jkl and mno are too small to move budget on their own.
- **Power limitations:** the power simulation assumes a 15% day-to-day spread in CPA. The real spread is 45% to 63% (3 to 4 times larger), so the true power is lower and "not significant" here means **not enough data**, not "no difference".

## Next steps
1. Agree on the smallest CPA gap worth acting on (for example 10%) **before** the next test.
2. Launch the abc scale-up test with a fixed budget cap and a fixed duration. Review it weekly.
3. To separate ghi, jkl and mno, run a planned test: shift a fixed share of budget between them for a set period and compare against a control.
4. Re-run this analysis monthly with fresh data and re-allocate based on BH-corrected results and confidence intervals.
